Obtain ID's of records that have zero subjects.  
Then free to experiment with methods to enrich their metadata.

In [107]:
NO_RECORDS = 1500

In [108]:
from datetime import datetime, timezone
from json import JSONDecodeError
from logging.config import dictConfig
from typing import Any, Optional
import psycopg
from psycopg import errors as psycopg_errors
from psycopg.rows import dict_row
from config.logging_config import LOGGING_CONFIG
from config.postgres_config import PostgresConfig
from tasks import transform_batch, add_file_metadata
import os
from fastapi import FastAPI, Query, HTTPException
import logging
from pydantic import BaseModel, Field
from utils.queue_utils import HarvestEventQueue, detect_identifier_type

from lxml import etree as ET
# API calls
import requests


# added for these experiments
from utils.enrichment_methods import get_doi, enrich_subjects_from_openaire

In [109]:
dictConfig(LOGGING_CONFIG)
logger = logging.getLogger(__name__)

In [110]:
user = os.getenv("POSTGRES_USER")
passwd = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_ADDRESS", "127.0.0.1")

if host == '':
    host = "127.0.0.1"

connection_params = {
    "user" : user,
    "password" : passwd,
    "dbname" : "datasetdb",
    "port" : 5432,
    "host" : host
}

In [111]:
query_records = """
            -- Exploring subjects from record raw metadata

-- Every xpath defines a potential XML structure with corresponding subject tag placement.
-- If raw_metadata conforms to ANY of these xpaths, subjects will be concatenated <subject> tags. (removing array_to_string() will return arrays)
-- If raw_metadata doesn't conform to any path (and probably has no subject tags), subjects will be ''


WITH query AS (
    SELECT
        records.*,
        array_to_string(
                xpath('oai:record/oai:metadata/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oaidc:oai_datacite/oaidc:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/},
                      {oaidc, http://schema.datacite.org/oai/oai-1.1/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oai:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
				-- note the 3 in datacite
                    '{{datacite, http://datacite.org/schema/kernel-3},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[] ||
				-- same path as previous but uses kernel-4 namespace
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[]
				, ', '
        ) AS subjects
    FROM public.records
)
SELECT 
    id,
    title,
    raw_metadata,
    subjects
FROM query
WHERE query.subjects = ''
LIMIT %s
;
"""

In [112]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_records, (NO_RECORDS,))
        
        results = cur.fetchall()
        print(results)

        record_ids = [str(row["id"]) for row in results]
        record_titles = [str(row["title"]) for row in  results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

[{'id': '8393ec5c-7723-4f4e-9d63-f27fcad32e36::doi:10.34894/VLN5PZ', 'title': 'Clinical relevance of verbal memory impairment in polydrug Ecstasy users: A pooled data analysis', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>doi:10.34894/VLN5PZ</identifier>\n    <datestamp>2025-07-05T00:00:18Z</datestamp>\n    <setSpec>MaastrichtUniversity</setSpec>\n    <setSpec>dataversenl</setSpec>\n  </header>\n  <metadata>\n    <resource xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.1/metadata.xsd">\n      <identifier identifierType="DOI">10.34894/VLN5PZ</identifier>\n      <creators>\n        <creator>\n          <creatorName nameType="Personal">Kuypers, K. P. C</creatorName>\n          <givenName>K. P. C</givenName>\n          <familyName>Kuype

In [113]:
record_titles

['Clinical relevance of verbal memory impairment in polydrug Ecstasy users: A pooled data analysis',
 'Dataset_Commons',
 'Sozialisationsprozesse beim Übergang in den Lehrbetrieb (SoLe) - 2011-2017',
 'Survey among Swiss Abroad',
 'Situation professionnelle deux ans après des licenciements collectifs dans le secteur industriel - 2011',
 'Umfrage: Immigrantinnen und Immigranten in der Schweiz-2013',
 'Widely Covering Data Collections/Database']

In [114]:
record_raw_metadata[2]

'<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:oai="http://www.openarchives.org/OAI/2.0/" xmlns:datacite="http://datacite.org/schema/kernel-4">\n  <header>\n    <identifier>oai:swissubase.ch:dcb4c8fc-f69e-49d2-a7cb-48b9f7c007fa</identifier>\n    <datestamp>2026-03-24T12:04:32Z</datestamp>\n  </header>\n  <metadata>\n    <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 https://schema.datacite.org/meta/kernel-4.6/metadata.xsd">\n      <datacite:identifier identifierType="DOI">10.23662/FORS-DS-686-1</datacite:identifier>\n      <datacite:creators>\n        <datacite:creator>\n          <datacite:creatorName nameType="Personal">Neuenschwander, Markus P.</datacite:creatorName>\n        </datacite:creator>\n      </datacite:creators>\n      <datacite:titles>\n        <datacite:title xml:lang="de">Sozialisationsprozesse beim Übergang in den Lehrbetrieb (SoLe) - 2011-

In [115]:
get_doi(record_raw_metadata[2])

'10.23662/FORS-DS-686-1'

# DOI check through bigger repositories  (#1)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Seaches thru OpenAlex or Openaire - apis?
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword

OpenAIRE API
- they're officialy moving to a Graph API in May 2026
https://api.openaire.eu/graph/swagger-ui/index.html

In [116]:
import logging
logging.getLogger("urllib3").setLevel(logging.WARNING)

In [117]:
doi = get_doi(record_raw_metadata[0]) 

if doi == []:
    print("NO DOI")
else:
    print(enrich_subjects_from_openaire(doi))

[]


In [118]:
# get only records that have a DOI
ix_has_doi = []

for i in range(0, len(record_raw_metadata)):
    rec = get_doi(record_raw_metadata[i])
    if doi != []:
        ix_has_doi.append(i)

print(ix_has_doi)

[0, 1, 2, 3, 4, 5, 6]


In [119]:
id_to_enrich = [record_ids[i] for i in ix_has_doi]
doi_to_enrich = [get_doi(record_raw_metadata[i]) for i in ix_has_doi]

print(id_to_enrich)
print(doi_to_enrich)

['8393ec5c-7723-4f4e-9d63-f27fcad32e36::doi:10.34894/VLN5PZ', '8393ec5c-7723-4f4e-9d63-f27fcad32e36::doi:10.34894/JESW3L', '8409b435-a7a8-4527-a955-436ad8d92b85::dcb4c8fc-f69e-49d2-a7cb-48b9f7c007fa', '8409b435-a7a8-4527-a955-436ad8d92b85::6e4b1efa-9e7d-43bb-86e5-8ffcfbca885d', '8409b435-a7a8-4527-a955-436ad8d92b85::38b31716-8dbc-4c90-8aff-2b92e923a499', '8409b435-a7a8-4527-a955-436ad8d92b85::3557f168-dfca-4bc3-91b4-8d5ea39f902b', '8393ec5c-7723-4f4e-9d63-f27fcad32e36::doi:10.34894/7LUSNC']
['10.34894/VLN5PZ', '10.34894/JESW3L', '10.23662/FORS-DS-686-1', '10.23662/FORS-DS-690-1', '10.23662/FORS-DS-675-3', '10.23662/FORS-DS-692-3', '10.34894/7LUSNC']


In [120]:
# limit to smaller section
#id_to_enrich = id_to_enrich[:30]
#doi_to_enrich = doi_to_enrich[:30]

In [121]:
from utils.enrichment_methods import _openaire_batch_doi_resolve, _opeanaire_get_doi_subjects, _ids_connect_with_openaire_subjects, batch_enrich_subjects_from_openaire

api_result = _openaire_batch_doi_resolve(doi_to_enrich)

print(api_result[:3])

[{'authors': [{'fullName': 'Blatter, Joachim', 'name': 'Joachim', 'surname': 'Blatter', 'rank': 1, 'pid': None}, {'fullName': 'Schlenker, Andrea', 'name': 'Andrea', 'surname': 'Schlenker', 'rank': 2, 'pid': None}], 'openAccessColor': None, 'publiclyFunded': None, 'type': 'dataset', 'language': {'code': 'und', 'label': 'Undetermined'}, 'countries': None, 'subjects': None, 'mainTitle': 'Survey: Immigrants in Switzerland', 'subTitle': None, 'descriptions': None, 'publicationDate': '2018-01-01', 'publisher': None, 'embargoEndDate': None, 'sources': None, 'formats': None, 'contributors': None, 'coverages': None, 'bestAccessRight': None, 'container': None, 'documentationUrls': None, 'codeRepositoryUrl': None, 'programmingLanguage': None, 'contactPeople': None, 'contactGroups': None, 'tools': None, 'size': None, 'version': None, 'geoLocations': None, 'id': 'doi_________::483e28968d7f212c6be0cee910e4b794', 'originalIds': ['50|datacite____::483e28968d7f212c6be0cee910e4b794', '10.23662/fors-ds-6

In [122]:
doi_subjects = _opeanaire_get_doi_subjects(api_result)
doi_subjects

[{'doi': '10.23662/fors-ds-686-1',
  'subjects': ['4. Education',
   '8. Economic growth',
   '10. No inequality',
   '3. Good health']},
 {'doi': '10.48573/00a2-vw63',
  'subjects': ['4. Education',
   '8. Economic growth',
   '10. No inequality',
   '3. Good health']},
 {'doi': '10.48573/c09h-h441', 'subjects': ['8. Economic growth']},
 {'doi': '10.48573/zset-sh38', 'subjects': ['8. Economic growth']},
 {'doi': '10.23662/fors-ds-675-3', 'subjects': ['8. Economic growth']},
 {'doi': '10.48573/2vz5-bh60', 'subjects': ['8. Economic growth']}]

In [123]:
id_dois = []
for ix in range(len(id_to_enrich)):
    id_dois.append(
        {
            'id': id_to_enrich[ix],
            'doi': doi_to_enrich[ix].lower(),
        }
    )
id_dois

[{'id': '8393ec5c-7723-4f4e-9d63-f27fcad32e36::doi:10.34894/VLN5PZ',
  'doi': '10.34894/vln5pz'},
 {'id': '8393ec5c-7723-4f4e-9d63-f27fcad32e36::doi:10.34894/JESW3L',
  'doi': '10.34894/jesw3l'},
 {'id': '8409b435-a7a8-4527-a955-436ad8d92b85::dcb4c8fc-f69e-49d2-a7cb-48b9f7c007fa',
  'doi': '10.23662/fors-ds-686-1'},
 {'id': '8409b435-a7a8-4527-a955-436ad8d92b85::6e4b1efa-9e7d-43bb-86e5-8ffcfbca885d',
  'doi': '10.23662/fors-ds-690-1'},
 {'id': '8409b435-a7a8-4527-a955-436ad8d92b85::38b31716-8dbc-4c90-8aff-2b92e923a499',
  'doi': '10.23662/fors-ds-675-3'},
 {'id': '8409b435-a7a8-4527-a955-436ad8d92b85::3557f168-dfca-4bc3-91b4-8d5ea39f902b',
  'doi': '10.23662/fors-ds-692-3'},
 {'id': '8393ec5c-7723-4f4e-9d63-f27fcad32e36::doi:10.34894/7LUSNC',
  'doi': '10.34894/7lusnc'}]

In [124]:
enriched_list = _ids_connect_with_openaire_subjects(id_dois, doi_subjects)
enriched_list

[{'id': '8393ec5c-7723-4f4e-9d63-f27fcad32e36::doi:10.34894/VLN5PZ',
  'doi': '10.34894/vln5pz',
  'enriched_subjects': None},
 {'id': '8393ec5c-7723-4f4e-9d63-f27fcad32e36::doi:10.34894/JESW3L',
  'doi': '10.34894/jesw3l',
  'enriched_subjects': None},
 {'id': '8409b435-a7a8-4527-a955-436ad8d92b85::dcb4c8fc-f69e-49d2-a7cb-48b9f7c007fa',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': {'10. No inequality',
   '3. Good health',
   '4. Education',
   '8. Economic growth'}},
 {'id': '8409b435-a7a8-4527-a955-436ad8d92b85::6e4b1efa-9e7d-43bb-86e5-8ffcfbca885d',
  'doi': '10.23662/fors-ds-690-1',
  'enriched_subjects': None},
 {'id': '8409b435-a7a8-4527-a955-436ad8d92b85::38b31716-8dbc-4c90-8aff-2b92e923a499',
  'doi': '10.23662/fors-ds-675-3',
  'enriched_subjects': {'8. Economic growth'}},
 {'id': '8409b435-a7a8-4527-a955-436ad8d92b85::3557f168-dfca-4bc3-91b4-8d5ea39f902b',
  'doi': '10.23662/fors-ds-692-3',
  'enriched_subjects': None},
 {'id': '8393ec5c-7723-4f4e-9d63-f27fcad32

In [125]:
# Only succesfully enriched IDs and subjects.
# Could be useful for comparing how many ids we sent to be enriched vs
# how many actually got new subjects
[el for el in enriched_list if el['enriched_subjects'] is not None]

[{'id': '8409b435-a7a8-4527-a955-436ad8d92b85::dcb4c8fc-f69e-49d2-a7cb-48b9f7c007fa',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': {'10. No inequality',
   '3. Good health',
   '4. Education',
   '8. Economic growth'}},
 {'id': '8409b435-a7a8-4527-a955-436ad8d92b85::38b31716-8dbc-4c90-8aff-2b92e923a499',
  'doi': '10.23662/fors-ds-675-3',
  'enriched_subjects': {'8. Economic growth'}}]

In [126]:
# All-in-one function

batch_enrich_subjects_from_openaire(id_dois)

[{'id': '8393ec5c-7723-4f4e-9d63-f27fcad32e36::doi:10.34894/VLN5PZ',
  'doi': '10.34894/vln5pz',
  'enriched_subjects': None},
 {'id': '8393ec5c-7723-4f4e-9d63-f27fcad32e36::doi:10.34894/JESW3L',
  'doi': '10.34894/jesw3l',
  'enriched_subjects': None},
 {'id': '8409b435-a7a8-4527-a955-436ad8d92b85::dcb4c8fc-f69e-49d2-a7cb-48b9f7c007fa',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': {'10. No inequality',
   '3. Good health',
   '4. Education',
   '8. Economic growth'}},
 {'id': '8409b435-a7a8-4527-a955-436ad8d92b85::6e4b1efa-9e7d-43bb-86e5-8ffcfbca885d',
  'doi': '10.23662/fors-ds-690-1',
  'enriched_subjects': None},
 {'id': '8409b435-a7a8-4527-a955-436ad8d92b85::38b31716-8dbc-4c90-8aff-2b92e923a499',
  'doi': '10.23662/fors-ds-675-3',
  'enriched_subjects': {'8. Economic growth'}},
 {'id': '8409b435-a7a8-4527-a955-436ad8d92b85::3557f168-dfca-4bc3-91b4-8d5ea39f902b',
  'doi': '10.23662/fors-ds-692-3',
  'enriched_subjects': None},
 {'id': '8393ec5c-7723-4f4e-9d63-f27fcad32

In [127]:
# Comparison: Number of initial records vs number of records that have gained subjects via this method
enriched = batch_enrich_subjects_from_openaire(id_dois)
print(len([el['doi'] for el in enriched]))
print(len([el['doi'] for el in enriched if el['enriched_subjects'] is not None]))

7
2


In [128]:
# Taking a look at the metadata initially obtained from DB, subset of few records
full_info_enriched = [el for el in enriched if el['enriched_subjects'] is not None]

first_enriched = full_info_enriched[:10]

In [129]:
# indexes that have been enriched
ix_original = [record_ids.index(rec['id']) for rec in first_enriched]
ix_original

[2, 4]

In [130]:
first_enriched

[{'id': '8409b435-a7a8-4527-a955-436ad8d92b85::dcb4c8fc-f69e-49d2-a7cb-48b9f7c007fa',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': {'10. No inequality',
   '3. Good health',
   '4. Education',
   '8. Economic growth'}},
 {'id': '8409b435-a7a8-4527-a955-436ad8d92b85::38b31716-8dbc-4c90-8aff-2b92e923a499',
  'doi': '10.23662/fors-ds-675-3',
  'enriched_subjects': {'8. Economic growth'}}]

In [131]:
[record_raw_metadata[i] for i in ix_original]

['<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:oai="http://www.openarchives.org/OAI/2.0/" xmlns:datacite="http://datacite.org/schema/kernel-4">\n  <header>\n    <identifier>oai:swissubase.ch:dcb4c8fc-f69e-49d2-a7cb-48b9f7c007fa</identifier>\n    <datestamp>2026-03-24T12:04:32Z</datestamp>\n  </header>\n  <metadata>\n    <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 https://schema.datacite.org/meta/kernel-4.6/metadata.xsd">\n      <datacite:identifier identifierType="DOI">10.23662/FORS-DS-686-1</datacite:identifier>\n      <datacite:creators>\n        <datacite:creator>\n          <datacite:creatorName nameType="Personal">Neuenschwander, Markus P.</datacite:creatorName>\n        </datacite:creator>\n      </datacite:creators>\n      <datacite:titles>\n        <datacite:title xml:lang="de">Sozialisationsprozesse beim Übergang in den Lehrbetrieb (SoLe) - 2011

## OpenAlex API
https://developers.openalex.org/
https://developers.openalex.org/guides/authentication

Get multiple works by DOI

use | between DOIs, up to 100 DOIs at once!!
check out how the results look then.

https://api.openalex.org/works?filter=doi:https://doi.org/10.1234/a|https://doi.org/10.1234/b|https://doi.org/10.1234/c&per_page=100



Use the select parameter to return only specific fields:
https://api.openalex.org/works?select=id,doi,display_name

In [132]:
OpenAlex_API_key = os.getenv("OPENALEX_API_KEY")

In [133]:
OPENALEX_API_URL = "https://api.openalex.org/works/"

doi = '10.15151/esrf-es-1473312981'

# We have to append the ID directly in the path
query_url = OPENALEX_API_URL + "https://doi.org/" + doi

response = requests.get(
    query_url,
    params={"api_key": OpenAlex_API_key}, 
    timeout=30
)

data = response.json()


In [134]:
data

{'id': 'https://openalex.org/W6886229830',
 'doi': 'https://doi.org/10.15151/esrf-es-1473312981',
 'title': 'SAXS WAXS measurements on reference samples for different sample detector distances. Control of transmission measurements',
 'display_name': 'SAXS WAXS measurements on reference samples for different sample detector distances. Control of transmission measurements',
 'publication_year': 2024,
 'publication_date': '2024-01-25',
 'ids': {'openalex': 'https://openalex.org/W6886229830',
  'doi': 'https://doi.org/10.15151/esrf-es-1473312981'},
 'language': 'en',
 'primary_location': {'id': 'doi:10.15151/esrf-es-1473312981',
  'is_oa': True,
  'landing_page_url': 'https://doi.org/10.15151/esrf-es-1473312981',
  'pdf_url': None,
  'source': {'id': 'https://openalex.org/S7407051764',
   'display_name': 'European Synchrotron Radiation Facility',
   'issn_l': None,
   'issn': [],
   'is_oa': False,
   'is_in_doaj': False,
   'is_core': False,
   'host_organization': None,
   'host_organiza

In [135]:
oa_DOIS = [
    '10.15151/esrf-es-1001518064',
    '10.15151/esrf-es-2323561777',
    '10.15151/esrf-es-2323561778',
    '10.15151/esrf-es-2323561877',
    'https://doi.org/10.5286/ISIS.E.100051344',
    '10.34894/AUUSS2',
    '10.17026/DANS-28H-C53V',
    '10.34894/VLSN1K',
    '10.15151/ESRF-ES-1030397492',
    '10.17026/AR/DMHU53',
    '10.17026/AR/GHYQTO',
    '10.17026/DANS-ZU2-HF3N',
    '10.17026/DANS-ZTM-7SA5',
    '10.17026/DANS-ZUK-HN2Z',
    'https://doi.org/10.1016/j.paid.2021.110888',
    '10.23662/fors-ds-686-1'
]

In [136]:
OPENALEX_API_URL = "https://api.openalex.org/works"

def _openalex_batch_doi_resolve(dois, api_key):
    all_results = []
    chunk_size = 100

    # Strip any existing https://doi.org/ or http://doi.org/ prefix
    # so we can add it back consistently for every DOI
    clean_dois = [
        doi.replace("https://doi.org/", "")
           .replace("http://doi.org/", "")
        for doi in dois
    ]

    for i in range(0, len(clean_dois), chunk_size):
        chunk = clean_dois[i : i + chunk_size]

        # Join with pipe — each DOI gets the full https://doi.org/ prefix
        doi_filter = "|".join(f"https://doi.org/{doi}" for doi in chunk)
        # Build URL as f-string to avoid requests encoding the colons and pipes
        url = f"{OPENALEX_API_URL}?filter=doi:{doi_filter}&per_page={chunk_size}&api_key={api_key}"

        response = requests.get(url, timeout=30)
        response.raise_for_status()
        all_results.extend(response.json().get("results", []))

    return all_results

In [137]:
results = _openalex_batch_doi_resolve(oa_DOIS,
                    api_key=OpenAlex_API_key)
print(len(results))
results

2


[{'id': 'https://openalex.org/W3152931024',
  'doi': 'https://doi.org/10.1016/j.paid.2021.110888',
  'title': 'Personality, anxiety, and cognitive failures as predictors of flow proneness',
  'display_name': 'Personality, anxiety, and cognitive failures as predictors of flow proneness',
  'publication_year': 2021,
  'publication_date': '2021-04-08',
  'ids': {'openalex': 'https://openalex.org/W3152931024',
   'doi': 'https://doi.org/10.1016/j.paid.2021.110888',
   'mag': '3152931024'},
  'language': 'en',
  'primary_location': {'id': 'doi:10.1016/j.paid.2021.110888',
   'is_oa': False,
   'landing_page_url': 'https://doi.org/10.1016/j.paid.2021.110888',
   'pdf_url': None,
   'source': {'id': 'https://openalex.org/S151331055',
    'display_name': 'Personality and Individual Differences',
    'issn_l': '0191-8869',
    'issn': ['0191-8869', '1873-3549'],
    'is_oa': False,
    'is_in_doaj': False,
    'is_core': True,
    'host_organization': 'https://openalex.org/P4310320990',
    'ho

In [138]:
# Get potential subjects, two relevant fields: keywords and concepts
# Concepts is an older system, keywords is newer - but concepts is more diverse

In [139]:
kw_lists = [rez['keywords'] for rez in results]

keywords = [[item['display_name'] for item in sublist] for sublist in kw_lists]

keywords

[['Psychology',
  'Conscientiousness',
  'Extraversion and introversion',
  'Neuroticism',
  'Openness to experience',
  'Context (archaeology)',
  'Anxiety',
  'Personality',
  'Big Five personality traits',
  'Developmental psychology',
  'Clinical psychology',
  'Social psychology',
  'Psychiatry'],
 ['Cell',
  'Extracellular matrix',
  'Pathogenesis',
  'Protein folding',
  'Function (biology)',
  'Enzyme',
  'Cell adhesion',
  'Folding (DSP implementation)',
  'Gene']]

In [140]:
# Introduce a threshold of score:
keywords = [[item['display_name'] for item in sublist if item['score'] > 0.6] for sublist in kw_lists]

keywords

[['Psychology',
  'Conscientiousness',
  'Extraversion and introversion',
  'Neuroticism',
  'Openness to experience'],
 []]

In [146]:
def _opeanalex_get_doi_subjects(response: list[dict], score_treshold: float = 0.5):
    """
    Transform API response into a list of DOI-sets and their subject-sets.

    A single OpenAIRE ID will result in one or more DOIs and zero or more subjects.
    If is has no linked subjects, we can't enrich anything so we skip that ID entirely.
    The OpenAIRE ID itself doesn't concern us so it is not retained. 
    
    :param response: List[dict], the API response
    :return: List[dict]
        dict:
            DOI - DOI that is linked to at least one subject
            [Subjects] - all subjects for the given DOI
    """
    if response == []:
        # no DOIs with any results
        return []
    
    # sets of DOIs with relevant sets of subjects. Later, spread to individual DOIs
    enriched_dois = []
    enriched_subjects = []

    for el in response:
        kw_list = el['keywords']

        if (kw_list is None or kw_list == []):
            # this element doesn't contribute to enriching subjects,
            # so even the DOIs don't matter
            continue

        keywords = [item['display_name'] for item in kw_list if item['score'] > score_treshold]

        # After filtering by score it's possible that no keywords are left,
        # Then this DOI won't be enriched
        if not keywords:
            continue
        
        current_doi = el['doi']

        enriched_dois.append(current_doi)

        enriched_subjects.append(keywords)

    # Create a list of dicts as output

    doi_subjects = []

    for ix in range(len(enriched_dois)):
        doi_sub_pair = {'doi': doi,
                        'subjects': enriched_subjects[ix]}
        doi_subjects.append(doi_sub_pair)

    return doi_subjects

In [147]:
_opeanalex_get_doi_subjects(results)

[{'doi': '10.15151/esrf-es-1473312981',
  'subjects': ['Psychology',
   'Conscientiousness',
   'Extraversion and introversion',
   'Neuroticism',
   'Openness to experience',
   'Context (archaeology)',
   'Anxiety',
   'Personality',
   'Big Five personality traits']},
 {'doi': '10.15151/esrf-es-1473312981',
  'subjects': ['Cell',
   'Extracellular matrix',
   'Pathogenesis',
   'Protein folding',
   'Function (biology)']}]

In [148]:
_opeanalex_get_doi_subjects(results, score_treshold=0.65)

[{'doi': '10.15151/esrf-es-1473312981',
  'subjects': ['Psychology',
   'Conscientiousness',
   'Extraversion and introversion',
   'Neuroticism',
   'Openness to experience']}]

In [149]:
_opeanalex_get_doi_subjects(results, score_treshold=0)

[{'doi': '10.15151/esrf-es-1473312981',
  'subjects': ['Psychology',
   'Conscientiousness',
   'Extraversion and introversion',
   'Neuroticism',
   'Openness to experience',
   'Context (archaeology)',
   'Anxiety',
   'Personality',
   'Big Five personality traits',
   'Developmental psychology',
   'Clinical psychology',
   'Social psychology',
   'Psychiatry']},
 {'doi': '10.15151/esrf-es-1473312981',
  'subjects': ['Cell',
   'Extracellular matrix',
   'Pathogenesis',
   'Protein folding',
   'Function (biology)',
   'Enzyme',
   'Cell adhesion',
   'Folding (DSP implementation)',
   'Gene']}]

# DOI resolver method  (#2)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Talks to an API such as DataCite or SignPosting.
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword